# Task 2 - Filing Risk Extraction
Generate teacher-backed data, train QLoRA on a Colab T4/L4, then evaluate base and merged models on the identical source-disjoint test set. Set `USE_FIXTURE = True` only for an offline contract smoke run; fixture output is not assessment evidence.

In [ ]:
from pathlib import Path
import sys, os, asyncio
task_root = Path.cwd().parent if (Path.cwd().parent / 'src').exists() else Path.cwd() / 'task2_genai'
sys.path.insert(0, str(task_root.parent))
from task2_genai.src.dataset import generate_dataset, split_source_disjoint, to_chat_jsonl
USE_FIXTURE = False
teacher = None
if os.getenv('GROQ_API_KEY'):
    from task2_genai.src.teacher import GroqTeacher
    teacher = GroqTeacher()
    health = asyncio.run(teacher.health_check())
    print('teacher_health', health)
    if not health.get('available'):
        raise RuntimeError('Teacher capability/quota check failed; no fallback generation is permitted')
elif not USE_FIXTURE:
    raise RuntimeError('Set GROQ_API_KEY for assessment generation or explicitly set USE_FIXTURE=True for an offline smoke run')
examples, metadata = asyncio.run(generate_dataset(200, teacher=teacher))
if not metadata.get('complete'):
    raise RuntimeError(f"Only {metadata.get('count')} accepted examples were generated; inspect rejections before training")
splits = split_source_disjoint(examples)
print(metadata)
print({name: len(rows) for name, rows in splits.items()})
to_chat_jsonl(splits['train'], task_root / 'artifacts' / 'train.jsonl')
to_chat_jsonl(splits['validation'], task_root / 'artifacts' / 'validation.jsonl')
to_chat_jsonl(splits['test'], task_root / 'artifacts' / 'test.jsonl')

In [ ]:
from task2_genai.src.training import QLoRAConfig, train_qlora, write_training_config
config = QLoRAConfig()
write_training_config(task_root / 'artifacts' / 'training_config.json', config)
try:
    metrics = train_qlora(str(task_root / 'artifacts' / 'train.jsonl'), str(task_root / 'artifacts' / 'validation.jsonl'), str(task_root / 'artifacts' / 'qlora'), config)
    print(metrics)
except RuntimeError as exc:
    print('training_not_run:', exc)

In [ ]:
from task2_genai.src.training import load_model_for_evaluation
from task2_genai.src.evaluation import generate_model_outputs, evaluate_outputs, write_evaluation
merged_dir = task_root / 'artifacts' / 'qlora' / 'merged'
if merged_dir.exists():
    base_model, base_tokenizer = load_model_for_evaluation('Qwen/Qwen2.5-1.5B-Instruct')
    tuned_model, tuned_tokenizer = load_model_for_evaluation(str(merged_dir))
    base_outputs = generate_model_outputs(base_model, base_tokenizer, splits['test'])
    tuned_outputs = generate_model_outputs(tuned_model, tuned_tokenizer, splits['test'])
    evaluation = evaluate_outputs(splits['test'], base_outputs, tuned_outputs)
    write_evaluation(evaluation, task_root / 'artifacts' / 'evaluation.json')
    print(evaluation)
else:
    print('evaluation_not_run: merged checkpoint is not available')

## Manual review
Complete `artifacts/manual_review_template.csv` for ten held-out examples and report the optional Chroma fallback separately.